In [17]:
import sys
from pathlib import Path
import pandas as pd

src_path = Path("../src").resolve()
sys.path.insert(0, str(src_path))

RESULTS_DIR = Path("../outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
prices = pd.read_csv(
    "../data/processed/vic_spot_prices_2025_07.csv",
    parse_dates=["SETTLEMENTDATE"],
)

prices.head()

,SETTLEMENTDATE,REGIONID,RRP
0,2025-07-01 00:05:00,VIC1,176.61966
1,2025-07-01 00:10:00,VIC1,182.30989
2,2025-07-01 00:15:00,VIC1,169.47610
3,2025-07-01 00:20:00,VIC1,184.67607
4,2025-07-01 00:25:00,VIC1,185.73900


In [19]:
from optimisation import (
    optimise_battery,
    optimise_battery_milp,
)

In [20]:
lp_results = optimise_battery(prices)

lp_revenue = lp_results["revenue"].sum()

print(f"LP July revenue: ${lp_revenue:,.2f}")
print(lp_results["soc_mwh"].agg(["min", "max"]))

Solver status: Optimal
LP July revenue: $1,596,923.80
min     20.0
max    180.0
Name: soc_mwh, dtype: float64


In [21]:
lp_simultaneous = (
    (lp_results["charge_mw"] > 0.001)
    & (lp_results["discharge_mw"] > 0.001)
).sum()

print(
    "LP simultaneous charge/discharge intervals:",
    lp_simultaneous,
)

LP simultaneous charge/discharge intervals: 400


In [22]:
lp_results.to_csv(
    RESULTS_DIR / "lp_dispatch.csv",
    index=False
)

print("Saved lp_dispatch.csv")

Saved lp_dispatch.csv


In [23]:
milp_results = optimise_battery_milp(prices)

milp_revenue = milp_results["revenue"].sum()

print(f"MILP July revenue: ${milp_revenue:,.2f}")
print(milp_results["soc_mwh"].agg(["min", "max"]))

Solver status: Optimal
MILP July revenue: $1,594,736.38
min     20.0
max    180.0
Name: soc_mwh, dtype: float64


In [24]:
milp_simultaneous = (
    (milp_results["charge_mw"] > 0.001)
    & (milp_results["discharge_mw"] > 0.001)
).sum()

print(
    "MILP simultaneous charge/discharge intervals:",
    milp_simultaneous,
)

MILP simultaneous charge/discharge intervals: 0


In [25]:
milp_results.to_csv(
    RESULTS_DIR / "milp_dispatch.csv",
    index=False
)

print("Saved milp_dispatch.csv")

Saved milp_dispatch.csv


In [ ]:
baseline_results = pd.read_csv(
    RESULTS_DIR / "baseline_dispatch.csv",
    parse_dates=["SETTLEMENTDATE"],
)

summary = pd.DataFrame({
    "strategy": [
        "Rule-based",
        "LP",
        "MILP",
    ],
    "revenue": [
        baseline_results["revenue"].sum(),
        lp_results["revenue"].sum(),
        milp_results["revenue"].sum(),
    ],
    "min_soc_mwh": [
        baseline_results["soc_mwh"].min(),
        lp_results["soc_mwh"].min(),
        milp_results["soc_mwh"].min(),
    ],
    "max_soc_mwh": [
        baseline_results["soc_mwh"].max(),
        lp_results["soc_mwh"].max(),
        milp_results["soc_mwh"].max(),
    ],
    "simultaneous_intervals": [
        0,
        lp_simultaneous,
        milp_simultaneous,
    ],
})

summary.to_csv(
    RESULTS_DIR / "summary_metrics.csv",
    index=False
)


In [28]:
summary.style.format({
    "revenue": "${:,.2f}",
    "min_soc_mwh": "{:.1f}",
    "max_soc_mwh": "{:.1f}",
})

,strategy,revenue,min_soc_mwh,max_soc_mwh,simultaneous_intervals
0,Rule-based,"$521,874.69",20.0,180.0,0
1,LP,"$1,596,923.80",20.0,180.0,400
2,MILP,"$1,594,736.38",20.0,180.0,0
